# Neutrosophic πgγ-Topological Mapping Framework for Cybersecurity Risk Assessment and Threat Classification

**End-to-end implementation notebook** based on the supplied concept document.

The notebook implements the seven stages: dataset profiling; preprocessing; single-valued neutrosophic representation; finite data-induced topological construction; πgγ-inspired threat mapping; neutrosophic risk ranking; and experimental/mathematical validation.

## Important mathematical implementation note
The concept document names πgγ-open, πgγ-closed and πgγ-irresolute mappings, but it does **not** provide their formal set-theoretic definitions, axioms, or a specific πgγ-topology construction. Therefore, this notebook makes the computational assumptions explicit rather than presenting them as universally accepted definitions. It constructs a finite Alexandrov topology from a data-derived preorder and implements transparent, configurable finite-set preservation tests. Replace the marked definitions in Stage 4–5 with the exact definitions from the user's mathematical reference/paper before claiming formal theorem-level validation.

**No synthetic or dummy network records are generated.** The notebook requires the real UNSW-NB15 and/or CIC-IDS2017 CSV files. It supports either dataset alone or both. For large CIC files, use the sampling option only if memory is constrained; sampling is from the real records.

## 0. Environment and configuration

Run in Google Colab or a local Jupyter environment. In Colab, mount Google Drive and set `DATA_ROOT` to the folder containing the extracted dataset CSV files. The loader searches recursively.

In [ ]:
# Install only missing packages
%pip -q install numpy pandas scikit-learn scipy matplotlib seaborn psutil joblib

In [ ]:
import os, re, glob, gc, time, json, math, warnings, platform
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, LabelEncoder, label_binarize
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, confusion_matrix,
    classification_report, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
import joblib

SEED = 42
np.random.seed(SEED)

# Google Colab option
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# EDIT THIS: folder containing extracted UNSW-NB15 / CIC-IDS2017 CSV files
DATA_ROOT = '/content/drive/MyDrive/Cybersecurity_Datasets'

# Dataset selection: 'UNSW', 'CIC', or 'BOTH'
DATASET_MODE = 'BOTH'

# Real-record resource controls. None means use all records.
MAX_ROWS_PER_FILE = None       # e.g. 250000 for memory-limited execution
MAX_TOTAL_ROWS = None          # e.g. 1000000; None = no global cap
TEST_SIZE = 0.20
VAL_SIZE = 0.15                # fraction of the remaining training pool
K_NEIGHBORS = 8
TOPOLOGY_MAX_POINTS = 2500     # finite topology analysis is quadratic/exponential
RISK_TOP_K = 100
N_JOBS = -1

# Use a smaller finite subset only for topological set enumeration.
# The classifier and conventional metrics still use the complete train/test split.
print('Python:', platform.python_version())
print('RAM (GB):', round(psutil.virtual_memory().total/1e9, 2))
print('Dataset root exists:', Path(DATA_ROOT).exists(), DATA_ROOT)

## 1. Dataset acquisition and profiling

Place the original dataset CSV files under `DATA_ROOT`. Common file formats supported:
- UNSW-NB15: `UNSW_NB15_training-set.csv`, `UNSW_NB15_testing-set.csv`, or equivalent CSV exports.
- CIC-IDS2017: daily CSV flow exports such as `Monday-WorkingHours.pcap_ISCX.csv`.

The code infers labels from common columns (`label`, `Label`, `attack_cat`, `Attack`, etc.). It does not silently invent labels. Inspect the printed profile and adjust `LABEL_CANDIDATES` if your local export uses a different schema.

In [ ]:
LABEL_CANDIDATES = ['label', 'Label', 'attack_cat', 'Attack', ' attack_cat', 'class', 'Class']
ID_COLUMNS = {'flow_id', 'src_ip', 'dst_ip', 'source_ip', 'destination_ip',
              'timestamp', 'time', 'date', 'srcip', 'dstip', 'id'}

def find_csv_files(root):
    files = glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)
    return sorted([f for f in files if os.path.isfile(f)])

def identify_dataset(path):
    p = path.lower()
    if 'unsw' in p: return 'UNSW'
    if 'cic' in p or 'workinghours' in p or 'iscx' in p: return 'CIC'
    return 'UNKNOWN'

def normalize_label_value(v):
    if pd.isna(v): return np.nan
    s = str(v).strip()
    if s.lower() in {'0', 'benign', 'normal', 'background', 'normal traffic'}:
        return 'Normal'
    if s.lower() in {'1'}: return 'Attack'
    return s if s else np.nan

def choose_label_column(df):
    lower = {str(c).strip().lower(): c for c in df.columns}
    for cand in LABEL_CANDIDATES:
        key = cand.strip().lower()
        if key in lower: return lower[key]
    # fuzzy fallback only when a label-like field exists
    matches = [c for c in df.columns if any(k in str(c).lower() for k in ['label','attack_cat','class'])]
    return matches[0] if matches else None

def read_network_csv(path, max_rows=None):
    # Low-memory read; tolerate malformed lines in public dataset exports
    df = pd.read_csv(path, low_memory=False, nrows=max_rows, on_bad_lines='skip')
    df.columns = [str(c).strip() for c in df.columns]
    return df

csv_files = find_csv_files(DATA_ROOT)
if not csv_files:
    raise FileNotFoundError(
        f'No CSV files found under {DATA_ROOT}. Download/extract the real UNSW-NB15 or CIC-IDS2017 CSV files and update DATA_ROOT.'
    )

selected_files = []
for f in csv_files:
    ds = identify_dataset(f)
    if DATASET_MODE == 'BOTH' and ds in {'UNSW','CIC'}: selected_files.append(f)
    elif DATASET_MODE == ds: selected_files.append(f)
if not selected_files:
    raise FileNotFoundError(f'No files matching DATASET_MODE={DATASET_MODE}. Found {len(csv_files)} CSV files.')

print('Selected CSV files:', len(selected_files))
for f in selected_files[:30]: print(' -', f)
if len(selected_files) > 30: print(' ...')

In [ ]:
frames = []
for i, path in enumerate(selected_files):
    part = read_network_csv(path, MAX_ROWS_PER_FILE)
    label_col = choose_label_column(part)
    if label_col is None:
        print('SKIP (no recognized label column):', path)
        continue
    part = part.copy()
    part['_source_dataset'] = identify_dataset(path)
    part['_source_file'] = Path(path).name
    part['_raw_target'] = part[label_col].map(normalize_label_value)
    part = part.drop(columns=[label_col])
    part = part.dropna(subset=['_raw_target'])
    if MAX_TOTAL_ROWS is not None:
        remaining = MAX_TOTAL_ROWS - sum(len(x) for x in frames)
        if remaining <= 0: break
        part = part.iloc[:remaining]
    frames.append(part)
    print(f'Loaded {Path(path).name}: {len(part):,} labelled rows; target={label_col}')

if not frames:
    raise ValueError('No labelled data loaded. Inspect the CSV label-column names and update LABEL_CANDIDATES.')

raw_df = pd.concat(frames, ignore_index=True, sort=False)
del frames
gc.collect()

print('Combined shape:', raw_df.shape)
print('Target distribution:')
display(raw_df['_raw_target'].value_counts(dropna=False).to_frame('count'))
display(raw_df.head())

In [ ]:
# Stage 1: dataset profiling, attack characterization, class distribution
print('Rows, columns:', raw_df.shape)
print('Duplicate rows:', int(raw_df.duplicated().sum()))
profile = pd.DataFrame({
    'dtype': raw_df.dtypes.astype(str),
    'missing_count': raw_df.isna().sum(),
    'missing_pct': (raw_df.isna().mean()*100).round(2),
    'unique_count': raw_df.nunique(dropna=True)
})
display(profile.sort_values('missing_pct', ascending=False).head(30))

plt.figure(figsize=(12,5))
raw_df['_raw_target'].value_counts().head(30).plot(kind='bar')
plt.title('Attack-class distribution (top 30 labels)')
plt.xlabel('Traffic class'); plt.ylabel('Records'); plt.xticks(rotation=70); plt.tight_layout(); plt.show()

print('Dataset sources:')
display(raw_df['_source_dataset'].value_counts().to_frame('records'))

## 2. Preprocessing and feature preparation

- Remove exact duplicate observations.
- Convert infinite values to missing values.
- Drop columns that are entirely missing or constant.
- Keep source metadata out of the predictive feature matrix.
- Convert the raw attack label into a harmonized `Normal` versus named attack-category target. When both datasets are combined, category names are retained as observed; no unsupported cross-dataset equivalence is assumed.
- Split stratified into train, validation and test sets.
- Fit imputers, categorical encoders and robust scaling on training data only to prevent leakage.

In [ ]:
# Harmonize common benign/normal labels, retain attack names
df = raw_df.copy()
df['_target'] = df['_raw_target'].astype(str).str.strip()
df.loc[df['_target'].str.lower().isin(['benign','normal','background','normal traffic']), '_target'] = 'Normal'

# Remove metadata from model features, but retain source columns for auditing
metadata_cols = [c for c in ['_raw_target','_source_dataset','_source_file'] if c in df.columns]
y_text = df['_target'].copy()
X_raw = df.drop(columns=metadata_cols + ['_target'], errors='ignore').copy()

# Remove identifier-like columns and parse numeric-looking object columns
drop_cols = [c for c in X_raw.columns if c.strip().lower() in ID_COLUMNS]
X_raw = X_raw.drop(columns=drop_cols, errors='ignore')

for c in X_raw.columns:
    if X_raw[c].dtype == 'object':
        # Convert only if at least 98% of non-null values are numeric-convertible
        converted = pd.to_numeric(X_raw[c], errors='coerce')
        denom = max(int(X_raw[c].notna().sum()), 1)
        if converted.notna().sum()/denom >= 0.98:
            X_raw[c] = converted

X_raw = X_raw.replace([np.inf, -np.inf], np.nan)
all_missing = X_raw.columns[X_raw.isna().all()].tolist()
constant = [c for c in X_raw.columns if X_raw[c].nunique(dropna=True) <= 1]
X_raw = X_raw.drop(columns=list(set(all_missing + constant)), errors='ignore')

# Remove rows whose target is missing (normally already handled by loader)
valid = y_text.notna()
X_raw, y_text = X_raw.loc[valid].reset_index(drop=True), y_text.loc[valid].reset_index(drop=True)

# Avoid stratification failures for classes with fewer than 3 observations
counts = y_text.value_counts()
keep_classes = counts[counts >= 3].index
mask = y_text.isin(keep_classes)
X_raw, y_text = X_raw.loc[mask].reset_index(drop=True), y_text.loc[mask].reset_index(drop=True)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = list(label_encoder.classes_)
print('Feature matrix:', X_raw.shape)
print('Classes:', dict(enumerate(class_names)))
print('Removed all-missing/constant columns:', len(all_missing), len(constant))

In [ ]:
# Stratified train / validation / test split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train, test_size=VAL_SIZE, random_state=SEED, stratify=y_train
)

num_cols = X_train_raw.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train_raw.columns if c not in num_cols]

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float32)
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False, dtype=np.float32)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', ohe)
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('cat', categorical_pipe, cat_cols)
], remainder='drop', sparse_threshold=0)

X_train = np.asarray(preprocessor.fit_transform(X_train_raw), dtype=np.float32)
X_val = np.asarray(preprocessor.transform(X_val_raw), dtype=np.float32)
X_test = np.asarray(preprocessor.transform(X_test_raw), dtype=np.float32)

print('Train / val / test:', X_train.shape, X_val.shape, X_test.shape)
print('Class distribution train:', Counter(y_train))

## 3. Feature relevance analysis and baseline threat classifier

Feature relevance is estimated on the training partition only. The ExtraTrees classifier provides a strong, reproducible baseline; the later neutrosophic mapping and risk prioritization operate on its held-out predictions and calibrated probability-like outputs.

In [ ]:
# Mutual information on a bounded training subset for speed
mi_n = min(100000, len(X_train))
rng = np.random.default_rng(SEED)
mi_idx = rng.choice(len(X_train), size=mi_n, replace=False)
mi = mutual_info_classif(X_train[mi_idx], y_train[mi_idx], random_state=SEED)
feature_names = list(num_cols)
if cat_cols:
    try:
        cat_names = list(preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(cat_cols))
    except Exception:
        cat_names = [f'cat_{i}' for i in range(X_train.shape[1]-len(num_cols))]
    feature_names += cat_names
feature_names = feature_names[:X_train.shape[1]]
mi_df = pd.DataFrame({'feature': feature_names, 'mutual_information': mi}).sort_values('mutual_information', ascending=False)
display(mi_df.head(25))

plt.figure(figsize=(10,6))
sns.barplot(data=mi_df.head(20), x='mutual_information', y='feature')
plt.title('Top feature relevance (mutual information)')
plt.tight_layout(); plt.show()

In [ ]:
classifier = ExtraTreesClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=SEED, n_jobs=N_JOBS, min_samples_leaf=1
)
t0 = time.perf_counter()
classifier.fit(X_train, y_train)
fit_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
val_proba = classifier.predict_proba(X_val)
test_proba = classifier.predict_proba(X_test)
predict_seconds = time.perf_counter() - t0
y_pred = classifier.classes_[np.argmax(test_proba, axis=1)]

print('Fit time (s):', round(fit_seconds, 3))
print('Prediction time (s):', round(predict_seconds, 3))
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

## 4. Stage 3 — Single-valued neutrosophic network traffic representation

For each observation, derive:
- **Truth (T):** model evidence for its predicted class.
- **Indeterminacy (I):** uncertainty from normalized predictive entropy, augmented by the gap between the two largest class probabilities.
- **Falsity (F):** one minus truth.

These are operational membership scores in `[0,1]`, not ground-truth probabilities of philosophical truth/falsehood. They are kept separate and are not forced to sum to one.

In [ ]:
def neutrosophic_memberships(probabilities, eps=1e-12):
    p = np.clip(np.asarray(probabilities, dtype=float), eps, 1.0)
    p = p / p.sum(axis=1, keepdims=True)
    truth = p.max(axis=1)
    if p.shape[1] > 1:
        entropy = -(p*np.log(p)).sum(axis=1) / np.log(p.shape[1])
        sorted_p = np.sort(p, axis=1)
        margin_uncertainty = 1.0 - (sorted_p[:,-1] - sorted_p[:,-2])
        indeterminacy = np.clip(0.7*entropy + 0.3*margin_uncertainty, 0, 1)
    else:
        entropy = np.zeros(len(p))
        indeterminacy = 1.0 - truth
    falsity = 1.0 - truth
    return np.column_stack([truth, indeterminacy, falsity])

N_train = neutrosophic_memberships(classifier.predict_proba(X_train))
N_val = neutrosophic_memberships(val_proba)
N_test = neutrosophic_memberships(test_proba)

neutro_df = pd.DataFrame(N_test, columns=['truth_T','indeterminacy_I','falsity_F'])
neutro_df['predicted_class'] = [class_names[i] for i in y_pred]
neutro_df['actual_class'] = [class_names[i] for i in y_test]
display(neutro_df.head())

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(neutro_df['indeterminacy_I'], bins=30, kde=True)
plt.title('Indeterminacy distribution on test observations')
plt.xlabel('Indeterminacy membership (I)')
plt.tight_layout(); plt.show()

membership_summary = neutro_df[['truth_T','indeterminacy_I','falsity_F']].describe().T
display(membership_summary)
print('Mean T+I+F (not constrained to 1):',
      np.round(neutro_df[['truth_T','indeterminacy_I','falsity_F']].sum(axis=1).mean(), 4))

## 5. Stage 4 — Finite data-induced neutrosophic πgγ-topological space

A finite Alexandrov topology can be generated from a preorder. Here a directed relation `i ≤ j` is created when `j` is among the nearest neighbours of `i` in the three-dimensional `(T,I,F)` representation and is no less threat-evidential under the configured score. The reflexive-transitive closure yields a preorder; its upward-closed subsets form an Alexandrov topology.

Because enumerating all subsets grows exponentially, topology enumeration is restricted to `TOPOLOGY_MAX_POINTS` real test observations. A deterministic sample is used for this mathematical validation subtask only.

In [ ]:
def transitive_closure_bool(A):
    R = A.copy().astype(bool)
    np.fill_diagonal(R, True)
    n = len(R)
    for k in range(n):
        R |= (R[:, [k]] & R[[k], :])
    return R

def build_preorder(points, k=8):
    points = np.asarray(points, dtype=float)
    n = len(points)
    k_eff = min(max(1, k), max(1, n-1))
    nn = NearestNeighbors(n_neighbors=k_eff+1, metric='euclidean')
    nn.fit(points)
    distances, indices = nn.kneighbors(points)
    # Evidence score: truth + falsity - indeterminacy; directed tie-break is stable.
    score = points[:,0] + points[:,2] - points[:,1]
    R = np.eye(n, dtype=bool)
    for i in range(n):
        for j in indices[i,1:]:
            if score[j] >= score[i] - 1e-12:
                R[i,j] = True
    R = transitive_closure_bool(R)
    return R

n_top = min(TOPOLOGY_MAX_POINTS, len(N_test))
top_idx = np.random.default_rng(SEED).choice(len(N_test), size=n_top, replace=False)
N_top = N_test[top_idx]
y_top = y_test[top_idx]
preorder = build_preorder(N_top, K_NEIGHBORS)
print('Topology universe size:', n_top)
print('Preorder relation pairs:', int(preorder.sum()))
print('Reflexive:', bool(np.diag(preorder).all()))
print('Transitive:', bool(np.all((preorder[:,:,None] & preorder[None,:,:]) <= preorder[:,None,:])))

In [ ]:
# In a preorder, a set U is upward-closed iff x in U and x<=y implies y in U.
def is_upward_closed(mask, relation):
    members = np.flatnonzero(mask)
    if len(members) == 0: return True
    for x in members:
        if np.any(relation[x] & ~mask):
            return False
    return True

def enumerate_upsets(relation, max_n=16):
    n = len(relation)
    if n > max_n:
        raise ValueError(f'Exact topology enumeration requires n <= {max_n}; got {n}.')
    topology = []
    for bits in range(1 << n):
        mask = np.array([(bits >> i) & 1 for i in range(n)], dtype=bool)
        if is_upward_closed(mask, relation):
            topology.append(mask)
    return topology

# Enumerate exact topology on a smaller induced sub-universe, while retaining
# the larger relation for graph-based mapping/risk analysis.
N_AXIOM_POINTS = min(12, n_top)
R_axiom = preorder[:N_AXIOM_POINTS, :N_AXIOM_POINTS]
topology_sets = enumerate_upsets(R_axiom)
print('Exact finite Alexandrov topology size:', len(topology_sets))

def topology_axiom_report(topology, n):
    empty = np.zeros(n, dtype=bool)
    whole = np.ones(n, dtype=bool)
    contains_bounds = any(np.array_equal(s, empty) for s in topology) and any(np.array_equal(s, whole) for s in topology)
    union_ok = True
    intersection_ok = True
    # Verify all pairwise unions/intersections; finite topology is closed under arbitrary unions/intersections.
    lookup = {tuple(s.tolist()) for s in topology}
    for a in topology:
        for b in topology:
            union_ok &= tuple((a|b).tolist()) in lookup
            intersection_ok &= tuple((a&b).tolist()) in lookup
    return {'contains_empty_and_universe': contains_bounds,
            'finite_union_closure': bool(union_ok),
            'finite_intersection_closure': bool(intersection_ok),
            'topology_cardinality': len(topology)}
axiom_report = topology_axiom_report(topology_sets, N_AXIOM_POINTS)
display(pd.Series(axiom_report, name='verification'))

## 6. Stage 5 — πgγ-closed, πgγ-open and irresolute threat mapping

**Operational finite-set convention used in this notebook (must be replaced if the research paper defines these terms differently):**
1. The target space is the finite set of attack classes.
2. Each source observation maps to its predicted attack class.
3. A target subset is open/closed in the discrete target topology.
4. A source subset is open/closed in the constructed Alexandrov topology.
5. A mapping is open (closed) if the image of every source-open (source-closed) set is target-open (target-closed). An irresolute mapping is tested by requiring inverse images of target-open sets to be source-open.

The discrete target topology makes every target subset open and closed, so those two preservation checks are structurally satisfied. The inverse-image irresoluteness test remains informative. The πgγ labels are retained as framework names, but this finite operational test is **not a substitute for the formal πgγ definitions**.

In [ ]:
# Build source open sets for the exact small axiom universe
source_open = topology_sets
target_classes = np.unique(y_top)
target_n = len(class_names)

# Source-to-target mapping on topology validation universe
map_top = y_top.copy()

# Target discrete topology: all subsets are open and closed
def target_discrete_subsets(n):
    return [np.array([(bits >> i)&1 for i in range(n)], dtype=bool) for bits in range(1<<n)]

target_topology = target_discrete_subsets(target_n)

def image_of_mask(mask, mapping, n_target):
    out = np.zeros(n_target, dtype=bool)
    for idx in np.flatnonzero(mask):
        out[int(mapping[idx])] = True
    return out

def inverse_image_of_mask(target_mask, mapping):
    return np.array([bool(target_mask[int(v)]) for v in mapping], dtype=bool)

mapping_small = map_top[:N_AXIOM_POINTS]
closed_source = [~s for s in source_open]  # complement of an open set in a topology

pi_gamma_open_ok = True
for s in source_open:
    image = image_of_mask(s, mapping_small, target_n)
    pi_gamma_open_ok &= any(np.array_equal(image, t) for t in target_topology)

pi_gamma_closed_ok = True
for s in closed_source:
    image = image_of_mask(s, mapping_small, target_n)
    pi_gamma_closed_ok &= any(np.array_equal(image, t) for t in target_topology)

pi_gamma_irresolute_ok = True
for t in target_topology:
    pre = inverse_image_of_mask(t, mapping_small)
    pi_gamma_irresolute_ok &= is_upward_closed(pre, R_axiom)

mapping_checks = {
    'πgγ-closed mapping verification (discrete target convention)': bool(pi_gamma_closed_ok),
    'πgγ-open mapping verification (discrete target convention)': bool(pi_gamma_open_ok),
    'πgγ-irresolute inverse-image verification': bool(pi_gamma_irresolute_ok),
    'mapping composition consistency (identity composition)': True
}
display(pd.Series(mapping_checks, name='result'))

In [ ]:
# Counterexample-oriented check: find a target subset whose inverse image is not source-open
counterexample = None
for t in target_topology:
    pre = inverse_image_of_mask(t, mapping_small)
    if not is_upward_closed(pre, R_axiom):
        counterexample = {
            'target_subset_classes': [class_names[i] for i in np.flatnonzero(t)],
            'inverse_image_source_indices': np.flatnonzero(pre).tolist()
        }
        break
print('Irresoluteness counterexample:', counterexample if counterexample is not None else 'None found in enumerated finite space')

# Composition check for f: observations -> class IDs and identity g: classes -> classes
identity_target = np.arange(target_n)
composed = identity_target[mapping_small]
print('Composition equals original mapping:', bool(np.array_equal(composed, mapping_small)))

## 7. Stage 6 — Neutrosophic risk assessment and prioritization

Risk score combines:
- attack severity (transparent configurable mapping),
- attack frequency (inverse-frequency emphasis),
- truth membership,
- indeterminacy membership,
- falsity membership.

Risk levels are assigned using quantiles learned from the validation set, then applied to the test set. This avoids selecting thresholds from test labels. The risk score is a prioritization heuristic, not a calibrated estimate of financial loss or incident impact.

In [ ]:
# Severity defaults are ordinal heuristics; customize for your threat taxonomy.
SEVERITY_KEYWORDS = {
    'Normal': 0.05,
    'DoS': 0.85, 'DDoS': 0.90, 'Bot': 0.90, 'Backdoor': 0.95,
    'Shellcode': 0.95, 'Worm': 0.95, 'Ransomware': 1.00,
    'Infiltration': 0.95, 'Brute Force': 0.70, 'FTP-Patator': 0.70,
    'SSH-Patator': 0.70, 'PortScan': 0.45, 'Reconnaissance': 0.45,
    'Analysis': 0.40, 'Fuzzers': 0.60, 'Exploits': 0.85, 'Generic': 0.75
}
def severity_for_class(name):
    if name == 'Normal': return SEVERITY_KEYWORDS['Normal']
    for key, value in SEVERITY_KEYWORDS.items():
        if key.lower() in str(name).lower():
            return value
    return 0.65  # explicit fallback for unknown attack category

class_freq = np.bincount(y_train, minlength=len(class_names)).astype(float)
freq_norm = class_freq / max(class_freq.sum(), 1)
rarity = 1.0 - freq_norm
severity_vec = np.array([severity_for_class(n) for n in class_names])

def calculate_risk(prob, neutro):
    pred_class = np.argmax(prob, axis=1)
    severity = severity_vec[pred_class]
    rarity_component = rarity[pred_class]
    T, I, F = neutro.T
    score = (0.40*severity + 0.20*rarity_component +
             0.20*T + 0.15*I + 0.05*F)
    return score, severity, pred_class

val_risk, val_severity, val_pred = calculate_risk(val_proba, N_val)
test_risk, test_severity, test_pred = calculate_risk(test_proba, N_test)

risk_edges = np.quantile(val_risk, [0.25, 0.50, 0.80])
def risk_level(scores):
    return np.digitize(scores, risk_edges, right=True)  # 0 low, 1 medium, 2 high, 3 critical

risk_pred = risk_level(test_risk)
risk_val_pred = risk_level(val_risk)
risk_names = np.array(['Low','Medium','High','Critical'])
risk_results = pd.DataFrame({
    'actual_class': [class_names[i] for i in y_test],
    'predicted_class': [class_names[i] for i in test_pred],
    'truth_T': N_test[:,0], 'indeterminacy_I': N_test[:,1], 'falsity_F': N_test[:,2],
    'severity': test_severity, 'risk_score': test_risk,
    'risk_level': risk_names[risk_pred]
})
display(risk_results.sort_values('risk_score', ascending=False).head(20))

## 8. Stage 7 — Experimental analysis and validation metrics

The following includes the metrics explicitly listed in the concept document: Accuracy, Precision, Recall/detection rate, F1, Macro-F1, Weighted-F1, ROC-AUC, PR-AUC, Balanced Accuracy, MCC, FPR, FNR, per-class recall, neutrosophic representation summaries, mapping checks, risk metrics, and computational performance.

In [ ]:
def classification_metrics(y_true, y_hat, proba, labels):
    out = {
        'Accuracy': accuracy_score(y_true, y_hat),
        'Precision_macro': precision_score(y_true, y_hat, average='macro', zero_division=0),
        'Recall_macro': recall_score(y_true, y_hat, average='macro', zero_division=0),
        'F1_macro': f1_score(y_true, y_hat, average='macro', zero_division=0),
        'F1_weighted': f1_score(y_true, y_hat, average='weighted', zero_division=0),
        'Balanced_accuracy': balanced_accuracy_score(y_true, y_hat),
        'MCC': matthews_corrcoef(y_true, y_hat)
    }
    cm = confusion_matrix(y_true, y_hat, labels=np.arange(len(labels)))
    # Multiclass one-vs-rest aggregate FPR/FNR, micro-counted over class decisions
    fp = cm.sum(axis=0) - np.diag(cm)
    fn = cm.sum(axis=1) - np.diag(cm)
    tn = cm.sum() - (fp + fn + np.diag(cm))
    out['FPR_macro_ovr'] = np.mean(fp / np.maximum(fp+tn, 1))
    out['FNR_macro_ovr'] = np.mean(fn / np.maximum(fn+np.diag(cm), 1))
    try:
        if len(labels) == 2:
            out['ROC_AUC'] = roc_auc_score(y_true, proba[:,1])
            out['PR_AUC'] = average_precision_score(y_true, proba[:,1])
        else:
            Y = label_binarize(y_true, classes=np.arange(len(labels)))
            out['ROC_AUC_ovr_macro'] = roc_auc_score(Y, proba, average='macro', multi_class='ovr')
            out['PR_AUC_macro'] = average_precision_score(Y, proba, average='macro')
    except Exception as e:
        out['ROC_AUC'] = np.nan
        out['PR_AUC'] = np.nan
        print('AUC warning:', e)
    return out, cm

cls_metrics, cm = classification_metrics(y_test, y_pred, test_proba, class_names)
display(pd.DataFrame({'Metric': list(cls_metrics.keys()), 'Value': list(cls_metrics.values())}))

In [ ]:
plt.figure(figsize=(max(8, len(class_names)*0.7), max(6, len(class_names)*0.6)))
sns.heatmap(cm, annot=(len(class_names)<=15), fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Test confusion matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.xticks(rotation=70); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

per_class = classification_report(y_test, y_pred, labels=np.arange(len(class_names)),
                                  target_names=class_names, output_dict=True, zero_division=0)
per_class_df = pd.DataFrame(per_class).T
display(per_class_df)

In [ ]:
# Risk-level metrics: construct reference risk from true class severity and training rarity.
true_severity = np.array([severity_vec[i] for i in y_test])
true_rarity = rarity[y_test]
true_risk_reference = 0.70*true_severity + 0.30*true_rarity
risk_reference_edges = np.quantile(
    0.70*np.array([severity_vec[i] for i in y_val]) + 0.30*rarity[y_val],
    [0.25, 0.50, 0.80]
)
risk_true = np.digitize(true_risk_reference, risk_reference_edges, right=True)

risk_accuracy = accuracy_score(risk_true, risk_pred)
risk_macro_f1 = f1_score(risk_true, risk_pred, average='macro', zero_division=0)
high_critical = np.isin(risk_true, [2,3])
k = min(RISK_TOP_K, len(test_risk))
topk_idx = np.argsort(test_risk)[::-1][:k]
recall_at_k = (high_critical[topk_idx].sum() / max(high_critical.sum(), 1))
high_risk_recall = recall_score(high_critical.astype(int), np.isin(risk_pred,[2,3]).astype(int), zero_division=0)

risk_metrics = {
    'Risk classification accuracy': risk_accuracy,
    'Risk-level Macro-F1': risk_macro_f1,
    f'Recall@{k} (high/critical)': recall_at_k,
    'High-risk threat recall': high_risk_recall,
    'Risk ranking consistency (Spearman)': pd.Series(test_risk).corr(pd.Series(true_risk_reference), method='spearman'),
    'Risk assessment agreement (Cohen-like exact agreement)': np.mean(risk_pred == risk_true)
}
display(pd.Series(risk_metrics, name='value').to_frame())

In [ ]:
# Neutrosophic representation metrics
T, I, F = N_test.T
membership_consistency = np.mean((T + F) <= 1.0 + 1e-8)  # expected by F=1-T construction
uncertain_mask = I >= np.quantile(I, 0.75)
uncertainty_agreement = np.mean(y_pred[uncertain_mask] == y_test[uncertain_mask]) if uncertain_mask.any() else np.nan

# Sensitivity: change I by +/- 5%, recompute simple threat confidence/risk ranking.
I_plus = np.clip(I + 0.05, 0, 1)
I_minus = np.clip(I - 0.05, 0, 1)
risk_plus = 0.40*test_severity + 0.20*rarity[test_pred] + 0.20*T + 0.15*I_plus + 0.05*F
risk_minus = 0.40*test_severity + 0.20*rarity[test_pred] + 0.20*T + 0.15*I_minus + 0.05*F

neutro_metrics = {
    'Mean indeterminacy': float(I.mean()),
    'Median indeterminacy': float(np.median(I)),
    'High-indeterminacy fraction (top quartile)': float(uncertain_mask.mean()),
    'Membership consistency fraction (T+F<=1)': float(membership_consistency),
    'Uncertainty classification agreement': float(uncertainty_agreement),
    'Mean absolute risk sensitivity to I ±0.05': float(np.mean(np.abs(risk_plus-risk_minus)))
}
display(pd.Series(neutro_metrics, name='value').to_frame())

In [ ]:
# Computational performance and scalability
process = psutil.Process(os.getpid())
memory_mb = process.memory_info().rss / (1024**2)
n_test = len(X_test)
throughput = n_test / max(predict_seconds, 1e-12)

performance_metrics = {
    'Training rows': len(X_train),
    'Validation rows': len(X_val),
    'Testing rows': len(X_test),
    'Input feature count': X_train.shape[1],
    'Model fit time (seconds)': fit_seconds,
    'Prediction time (seconds)': predict_seconds,
    'Prediction throughput (records/second)': throughput,
    'Current process RSS memory (MB)': memory_mb,
    'Topology universe size': n_top,
    'Exact topology enumeration universe': N_AXIOM_POINTS,
    'Exact topology cardinality': len(topology_sets)
}
display(pd.Series(performance_metrics, name='value').to_frame())

# Simple scaling experiment on actual held-out records
scaling_rows = []
for frac in [0.10, 0.25, 0.50, 1.00]:
    n = max(1, int(len(X_test)*frac))
    t0 = time.perf_counter()
    _ = classifier.predict(X_test[:n])
    elapsed = time.perf_counter()-t0
    scaling_rows.append({'records': n, 'seconds': elapsed, 'records_per_second': n/max(elapsed,1e-12)})
scaling_df = pd.DataFrame(scaling_rows)
display(scaling_df)
plt.figure(figsize=(7,4))
plt.plot(scaling_df['records'], scaling_df['seconds'], marker='o')
plt.xlabel('Number of real test records'); plt.ylabel('Prediction time (s)')
plt.title('Computational scaling on held-out data'); plt.tight_layout(); plt.show()

## 9. Consolidated validation tables and visualizations

In [ ]:
all_metrics = {}
all_metrics.update({f'Classification | {k}': float(v) for k,v in cls_metrics.items() if np.isscalar(v)})
all_metrics.update({f'Neutrosophic | {k}': float(v) for k,v in neutro_metrics.items() if np.isscalar(v)})
all_metrics.update({f'Risk | {k}': float(v) for k,v in risk_metrics.items() if np.isscalar(v)})
all_metrics.update({f'Performance | {k}': float(v) for k,v in performance_metrics.items() if np.isscalar(v) and not isinstance(v,str)})
all_metrics.update({f'Mapping | {k}': float(v) for k,v in mapping_checks.items()})
all_metrics.update({f'Topology | {k}': float(v) for k,v in axiom_report.items() if isinstance(v,(bool,int,float))})

final_metrics_df = pd.DataFrame({'metric': list(all_metrics.keys()), 'value': list(all_metrics.values())})
display(final_metrics_df)

## 10. Export results (optional)

Exports metrics, risk-ranked alerts and a compact model pipeline to the output directory. No files are written unless `EXPORT_RESULTS=True`.

In [ ]:
EXPORT_RESULTS = False
OUTPUT_DIR = './neutrosophic_cybersecurity_outputs'

if EXPORT_RESULTS:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    final_metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'all_validation_metrics.csv'), index=False)
    risk_results.sort_values('risk_score', ascending=False).to_csv(
        os.path.join(OUTPUT_DIR, 'risk_ranked_test_alerts.csv'), index=False
    )
    mi_df.to_csv(os.path.join(OUTPUT_DIR, 'feature_relevance.csv'), index=False)
    pd.DataFrame({'class_id': range(len(class_names)), 'class_name': class_names}).to_csv(
        os.path.join(OUTPUT_DIR, 'class_mapping.csv'), index=False
    )
    joblib.dump({'preprocessor': preprocessor, 'classifier': classifier,
                 'label_encoder': label_encoder, 'risk_edges': risk_edges,
                 'severity_vec': severity_vec, 'rarity': rarity},
                os.path.join(OUTPUT_DIR, 'trained_framework.joblib'))
    with open(os.path.join(OUTPUT_DIR, 'run_summary.json'), 'w') as f:
        json.dump({
            'dataset_mode': DATASET_MODE,
            'selected_files': selected_files,
            'class_names': class_names,
            'configuration': {
                'seed': SEED, 'test_size': TEST_SIZE, 'validation_fraction': VAL_SIZE,
                'k_neighbors': K_NEIGHBORS, 'topology_max_points': TOPOLOGY_MAX_POINTS
            },
            'mapping_checks': mapping_checks,
            'topology_axioms': axiom_report
        }, f, indent=2)
    print('Exported to:', os.path.abspath(OUTPUT_DIR))
else:
    print('Export disabled. Set EXPORT_RESULTS=True to save outputs.')

## 11. Interpretation and reproducibility checklist

Before reporting results in a paper:
1. Record exact dataset release, source URL, file names, and any sampling settings.
2. Run UNSW-NB15 and CIC-IDS2017 independently as well as jointly; cross-dataset feature union can introduce dataset/source effects.
3. Review the inferred label mapping and remove any dataset-specific label ambiguity.
4. Replace the configurable severity table with a documented cybersecurity severity rubric or expert-labelled severity.
5. Replace the operational πgγ mapping conventions with the exact formal definitions and theorem assumptions adopted in the mathematical manuscript.
6. Report the finite topology universe size separately from the full classifier dataset size.
7. Do not interpret the risk score as calibrated incident loss without external severity/impact ground truth.

**Notebook outputs:** classification metrics; class-wise recall; confusion matrix; neutrosophic membership distributions; finite topology axiom checks; open/closed/irresolute mapping checks; counterexample search; risk-ranked alerts; Recall@K; processing time, memory and throughput.